In [2]:
"""
Quick test: does letting sampleRegions() pull a fine-resolution band via its
default (nearest-neighbor) behavior give a different answer than explicitly
reduceResolution()-ing that band onto the target grid first?

Single 1km test square, 2015 MODIS VCF (250m native) and 2015 RAP tree
cover (30m native), percent tree cover only, one band each.
"""

import ee

GEE_PROJECT = 'ee-tymc5571-multi-disturbance'

#Prepare to use Earth Engine
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.read_write',
        'https://www.googleapis.com/auth/drive'
    ]
)

ee.Initialize(project=GEE_PROJECT)

# ---------------------------------------------------------------------------
# small test area - reusing the Bailey, CO point from the main pipeline's
# commented-out test block, just buffered out to a ~1km square
# ---------------------------------------------------------------------------
test_pt = ee.Geometry.Point([-105.4733328, 39.4055449])
aoi = test_pt.buffer(500).bounds()

TARGET_CRS = 'EPSG:5070'
TARGET_SCALE = 250  # MODIS native scale - used as the common output grid
target_proj = ee.Projection(TARGET_CRS).atScale(TARGET_SCALE)

YEAR = 2015

# ---------------------------------------------------------------------------
# load single-year, single-band layers
# ---------------------------------------------------------------------------
modis_tree = (ee.ImageCollection('MODIS/061/MOD44B')
              .filter(ee.Filter.calendarRange(YEAR, YEAR, 'year'))
              .select('Percent_Tree_Cover')
              .first()
              .rename('modis_tree'))

rap_tree_raw = (ee.ImageCollection('projects/rap-data-365417/assets/vegetation-cover-v3')
                .filterDate(f'{YEAR}-01-01', f'{YEAR}-12-31')
                .select('TRE')
                .first()
                .rename('rap_tree'))

print('MODIS native scale (m):', modis_tree.projection().nominalScale().getInfo())
print('RAP native scale (m):', rap_tree_raw.projection().nominalScale().getInfo())

# ---------------------------------------------------------------------------
# TEST 1 - naive stack
# MODIS is band 1, already at the 250m target scale. RAP gets stacked in
# untouched at its native 30m grid - no reduceResolution anywhere.
# sampleRegions is called with no scale/projection args, so per the docs it
# falls back to band 1's (MODIS) defaults to build the sampling grid. The
# question is whether the RAP band gets aggregated onto that grid, or just
# nearest-neighbor picked.
# ---------------------------------------------------------------------------
naive_stack = ee.Image.cat([modis_tree, rap_tree_raw])

naive_sample = naive_stack.sampleRegions(
    collection=ee.FeatureCollection([ee.Feature(aoi)]),
    tileScale=1,
    geometries=True
)

# ---------------------------------------------------------------------------
# TEST 2 - RAP explicitly aggregated to the target grid first
# same MODIS band 1, but RAP is mean-reduced from 30m up to 250m/EPSG:5070
# before it's ever stacked, so its pixel grid already matches band 1
# ---------------------------------------------------------------------------
rap_tree_reduced = (rap_tree_raw
                     .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
                     .setDefaultProjection(target_proj)
                     .rename('rap_tree'))

reduced_stack = ee.Image.cat([modis_tree, rap_tree_reduced])

reduced_sample = reduced_stack.sampleRegions(
    collection=ee.FeatureCollection([ee.Feature(aoi)]),
    tileScale=1,
    geometries=True
)

# ---------------------------------------------------------------------------
# pull both small collections client-side and compare pixel by pixel
# same MODIS band drives the sampling grid in both cases, so the two
# collections should line up 1:1 - only the rap_tree value should differ
# ---------------------------------------------------------------------------
naive_feats = naive_sample.getInfo()['features']
reduced_feats = reduced_sample.getInfo()['features']

print(f'\n{len(naive_feats)} naive points, {len(reduced_feats)} reduced points\n')


def _key(feat):
    # round coords so float noise doesn't break the point matching
    x, y = feat['geometry']['coordinates']
    return (round(x, 5), round(y, 5))


naive_by_pt = {_key(f): f['properties'] for f in naive_feats}
reduced_by_pt = {_key(f): f['properties'] for f in reduced_feats}

print(f"{'lon':>12}{'lat':>12}{'modis':>10}{'rap_naive':>12}{'rap_reduced':>13}{'diff':>8}")
for key in sorted(naive_by_pt):
    naive_props = naive_by_pt[key]
    reduced_props = reduced_by_pt.get(key)
    if reduced_props is None:
        continue
    rap_naive = naive_props['rap_tree']
    rap_reduced = reduced_props['rap_tree']
    diff = rap_reduced - rap_naive
    print(f"{key[0]:>12.5f}{key[1]:>12.5f}{naive_props['modis_tree']:>10.1f}"
          f"{rap_naive:>12.2f}{rap_reduced:>13.2f}{diff:>8.2f}")


Successfully saved authorization token.
MODIS native scale (m): 231.65635826395828
RAP native scale (m): 30.000000000000004

18 naive points, 18 reduced points

         lon         lat     modis   rap_naive  rap_reduced    diff
  -105.47895    39.40521      16.0       60.00        33.48  -26.52
  -105.47850    39.40312      43.0       60.00        61.32    1.32
  -105.47717    39.40937      13.0       51.00        48.01   -2.99
  -105.47671    39.40729      16.0       13.00        36.09   23.09
  -105.47626    39.40521      15.0       66.00        28.99  -37.01
  -105.47580    39.40312      28.0       51.00        63.40   12.40
  -105.47447    39.40937      13.0       39.00        41.40    2.40
  -105.47401    39.40729      17.0       18.00        23.95    5.95
  -105.47356    39.40521       9.0       38.00         6.55  -31.45
  -105.47311    39.40312      42.0       38.00        63.83   25.83
  -105.47177    39.40937      21.0       55.00        50.80   -4.20
  -105.47132    39.407